In [1]:
# ============================================================
# 12_primary_model_leakagefree_FINAL.ipynb
# Block 1: Setup & Config
#
# Single regularised Gradient Boosting Survival Model.
# Fix vs all prior versions: Cox-Lasso expression gene selection
# now happens INSIDE each CV fold (previously fixed globally via
# NB03 pickle — this was a real selection leak, not just a
# theoretical one).
#
# No fusion/neural component. No ensemble blending.
# ============================================================

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter

# ---- Paths ----
base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
OUT_MODEL_DIR   = f'{base}/models/final_leakagefree_v2'
OUT_RESULTS_DIR = f'{base}/outputs/results'
os.makedirs(OUT_MODEL_DIR, exist_ok=True)
os.makedirs(OUT_RESULTS_DIR, exist_ok=True)

# ---- CV config ----
N_FOLDS   = 5
N_SEEDS   = 10                     # repeated CV for stability (Block 5)
SEEDS     = list(range(42, 42 + N_SEEDS))
N_DYSREG_SELECT = 20                # top-K dysreg genes per fold, by Cox p-value

# ---- Expression Lasso config (now fit INSIDE each fold) ----
# l1_ratio=1.0 -> pure Lasso within sksurv's Coxnet (elastic net) implementation
LASSO_L1_RATIO   = 1.0
LASSO_ALPHA_MIN_RATIO = 0.01
LASSO_N_ALPHAS   = 50
# We'll pick alpha via the model's internal CV path, but keep this
# explicit and inspectable rather than silently taking coef_[:, 0].

# ---- GBM config (unchanged from FINAL_primary_model_run.py) ----
GBM_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=2,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
)

# ---- Bootstrap config (Block 6) ----
N_BOOTSTRAP = 1000

print("Config loaded.")
print(f"  Folds: {N_FOLDS}, Seeds: {N_SEEDS} (total fits: {N_FOLDS * N_SEEDS})")
print(f"  Dysreg genes selected per fold: {N_DYSREG_SELECT}")
print(f"  GBM params: {GBM_PARAMS}")

Config loaded.
  Folds: 5, Seeds: 10 (total fits: 50)
  Dysreg genes selected per fold: 20
  GBM params: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 2, 'min_samples_split': 20, 'min_samples_leaf': 10, 'subsample': 0.8}


In [2]:
# ============================================================
# Block 2: Load & Align Data Streams
# ============================================================

print("[1/2] Loading data streams...")

expr     = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg   = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune   = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

common = expr.index.intersection(dysreg.index).intersection(
         immune.index).intersection(clinical.index)

expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

print(f"   Patients aligned across all 4 streams: {len(common)}")
print(f"   Expression shape:    {expr.shape}")
print(f"   Dysregulation shape: {dysreg.shape}")
print(f"   Immune shape:        {immune.shape}")
print(f"   Clinical shape:      {clinical.shape}")

# ------------------------------------------------------------
# [2/2] SANITY CHECKS — catch silent string-mismatch bugs
# before they quietly zero out a whole feature.
# ------------------------------------------------------------
print("\n[2/2] Categorical sanity checks...")

print("\n-- gender --")
print(clinical['gender'].value_counts(dropna=False))
print("   NOTE: code assumes lowercase 'male' string match.")
print("   If values above don't include exactly 'male', gender feature will be silently wrong.")

print("\n-- stage_group --")
print(clinical['stage_group'].value_counts(dropna=False))
print("   NOTE: code expects exact strings 'Stage I', 'Stage II', 'Stage III', 'Stage IV'.")
print("   If values above differ (e.g. 'III', 'stage_3'), dummies/interactions will break or silently misfire.")

print("\n-- event --")
print(clinical['event'].value_counts(dropna=False))
print("   NOTE: expected boolean-like (0/1 or True/False). Check for unexpected values (e.g. 2, NaN).")

print("\n-- survival_time --")
print(clinical['survival_time'].describe())
n_neg_or_zero = (clinical['survival_time'] <= 0).sum()
n_na = clinical['survival_time'].isna().sum()
print(f"   survival_time <= 0: {n_neg_or_zero}   (should be 0 per your doc's filtering)")
print(f"   survival_time NaN:  {n_na}   (should be 0)")

print("\n-- immune columns needed by interaction features --")
required_immune_cols = ['Macrophages M2', 'T cells CD8', 'T cells regulatory (Tregs)']
for col in required_immune_cols:
    status = "OK" if col in immune.columns else "MISSING — will KeyError later"
    print(f"   '{col}': {status}")

print("\n-- expression / dysreg NaN check --")
print(f"   Expression NaNs: {expr.isna().sum().sum()}")
print(f"   Dysreg NaNs:     {dysreg.isna().sum().sum()}")

[1/2] Loading data streams...
   Patients aligned across all 4 streams: 478
   Expression shape:    (478, 1000)
   Dysregulation shape: (478, 819)
   Immune shape:        (478, 22)
   Clinical shape:      (478, 10)

[2/2] Categorical sanity checks...

-- gender --
gender
female    257
male      221
Name: count, dtype: int64
   NOTE: code assumes lowercase 'male' string match.
   If values above don't include exactly 'male', gender feature will be silently wrong.

-- stage_group --
stage_group
Stage I      256
Stage II     112
Stage III     77
Stage IV      25
NaN            8
Name: count, dtype: int64
   NOTE: code expects exact strings 'Stage I', 'Stage II', 'Stage III', 'Stage IV'.
   If values above differ (e.g. 'III', 'stage_3'), dummies/interactions will break or silently misfire.

-- event --
event
0    357
1    121
Name: count, dtype: int64
   NOTE: expected boolean-like (0/1 or True/False). Check for unexpected values (e.g. 2, NaN).

-- survival_time --
count     478.000000
mea

In [3]:
# Check raw clinical file for recoverable staging info
clinical_raw = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)
missing_stage_ids = clinical_raw[clinical_raw['stage_group'].isna()].index.tolist()

print(f"Patients with missing stage_group: {len(missing_stage_ids)}")
print(missing_stage_ids)
print()
print("All columns in clinical file (looking for alternate staging fields):")
print(clinical_raw.columns.tolist())
print()
print("Raw rows for these patients (all columns):")
print(clinical_raw.loc[missing_stage_ids])

Patients with missing stage_group: 8
['tcga-38-4626', 'tcga-50-5045', 'tcga-55-5899', 'tcga-64-1678', 'tcga-67-4679', 'tcga-69-7765', 'tcga-69-8254', 'tcga-73-4677']

All columns in clinical file (looking for alternate staging fields):
['vital_status', 'days_to_death', 'days_to_last_followup', 'age', 'gender', 'stage', 'survival_time', 'event', 'stage_group', 'age_group']

Raw rows for these patients (all columns):
                            vital_status  days_to_death  \
patient.bcr_patient_barcode                               
tcga-38-4626                       alive            NaN   
tcga-50-5045                        dead         2174.0   
tcga-55-5899                       alive            NaN   
tcga-64-1678                       alive            NaN   
tcga-67-4679                       alive            NaN   
tcga-69-7765                       alive            NaN   
tcga-69-8254                       alive            NaN   
tcga-73-4677                        dead          

In [4]:
# ------------------------------------------------------------
# Drop patients with missing stage_group — confirmed unrecoverable
# (raw 'stage' column is also NaN for these 8; no alternate
# staging field exists in clinical_survival.csv)
# ------------------------------------------------------------
n_before = len(common)
valid_stage_ids = clinical[clinical['stage_group'].notna()].index
common = common.intersection(valid_stage_ids)
n_after = len(common)

print(f"Dropped {n_before - n_after} patients with missing stage_group.")
print(f"Cohort: {n_before} -> {n_after}")

# Re-slice all 4 streams to the corrected cohort
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

print(f"\nFinal aligned cohort: {len(common)} patients")
print(clinical['stage_group'].value_counts())
print(f"\nEvents: {clinical['event'].sum()} ({clinical['event'].mean()*100:.1f}%)")

Dropped 8 patients with missing stage_group.
Cohort: 478 -> 470

Final aligned cohort: 470 patients
stage_group
Stage I      256
Stage II     112
Stage III     77
Stage IV      25
Name: count, dtype: int64

Events: 119 (25.3%)


In [5]:
# ============================================================
# Block 3: Survival labels + fixed (non-selected) features
#
# These do NOT involve any label-based selection, so they're
# safe to compute once, globally, before the CV loop.
# (Contrast with expression/dysreg genes in Block 4, which
# MUST be selected inside each fold.)
# ============================================================

# ---- Survival label array (structured, for sksurv) ----
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"Cohort: {len(common)} patients")
print(f"Events: {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")
print(f"Survival time range: {y['time'].min():.0f} - {y['time'].max():.0f} days")

# ---- Clinical features (age, gender, stage dummies) ----
age    = clinical[['age']].copy()
gender = (clinical['gender'] == 'male').astype(float).to_frame()

stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
# Safe now: no NaNs left in stage_group after Block 2b drop.
# Sanity assert to make sure this stays true if data changes upstream:
assert clinical['stage_group'].isna().sum() == 0, "Unexpected NaN in stage_group post-filter"

clinical_features = pd.concat([age, gender, stage_dummies], axis=1).astype(float)

print(f"\nClinical features: {clinical_features.shape[1]} columns")
print(clinical_features.columns.tolist())
print(f"NaNs in clinical_features: {clinical_features.isna().sum().sum()} (should be 0)")

# ---- Immune features (all 22, used as-is, no selection) ----
immune_features = immune.copy()
print(f"\nImmune features: {immune_features.shape[1]} columns (unselected, used in full)")

# ---- Interaction features ----
# NOTE: these are hand-picked based on biological reasoning, not
# fit/searched on this data's labels, so they don't constitute a
# label-based leak. Flagging this explicitly for the paper's
# methods section: these 5 terms were pre-specified, not learned.
interactions = pd.DataFrame({
    'stageIII_x_M2':   (clinical_features['stage_Stage III'] *
                        immune_features['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_features['stage_Stage IV'] *
                        immune_features['T cells CD8']).values,
    'age_x_stageIII':  (clinical_features['age'] *
                        clinical_features['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_features['stage_Stage III'] *
                        immune_features['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_features['Macrophages M2'] *
                        immune_features['T cells CD8']).values,
}, index=common)

print(f"\nInteraction features: {interactions.shape[1]} columns")
for col in interactions.columns:
    print(f"   {col}: mean={interactions[col].mean():.4f}, std={interactions[col].std():.4f}")

# ---- Summary of fixed feature block sizes ----
n_fixed = clinical_features.shape[1] + immune_features.shape[1] + interactions.shape[1]
print(f"\nFixed features total (clinical + immune + interactions): {n_fixed}")
print("Remaining (selected inside fold, Block 4): 72 expression + 20 dysreg = 92")
print(f"Expected grand total: {n_fixed + 92}")

Cohort: 470 patients
Events: 119 (25.3%)
Survival time range: 1 - 6812 days

Clinical features: 5 columns
['age', 'gender', 'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']
NaNs in clinical_features: 0 (should be 0)

Immune features: 22 columns (unselected, used in full)

Interaction features: 5 columns
   stageIII_x_M2: mean=0.0212, std=0.0535
   stageIV_x_CD8: mean=0.0018, std=0.0133
   age_x_stageIII: mean=10.7596, std=24.6805
   stageIII_x_Treg: mean=0.0046, std=0.0176
   M2_x_CD8: mean=0.0047, std=0.0062

Fixed features total (clinical + immune + interactions): 32
Remaining (selected inside fold, Block 4): 72 expression + 20 dysreg = 92
Expected grand total: 124


In [6]:
# ============================================================
# Block 4: In-fold gene selection functions + single-seed
# sanity-check run.
#
# KEY FIX: expression genes now selected via Cox-Lasso INSIDE
# each fold (mirrors what dysreg selection already did
# correctly). No pre-fitted NB03 pickle is used anywhere below.
# ============================================================

def select_expression_genes_infold(expr_train, y_train, n_inner_folds=3):
    """
    Fit Cox-Lasso on TRAINING FOLD ONLY. Alpha chosen via inner
    CV on training data only (never touches the outer test fold).
    Returns list of selected gene names (nonzero coefficients).
    """
    coxnet = CoxnetSurvivalAnalysis(
        l1_ratio=LASSO_L1_RATIO,
        alpha_min_ratio=LASSO_ALPHA_MIN_RATIO,
        n_alphas=LASSO_N_ALPHAS,
        max_iter=100000,
    )
    coxnet.fit(expr_train.values, y_train)
    alphas = coxnet.alphas_

    # Inner CV to score each alpha on training data only
    inner_kf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=0)
    alpha_scores = np.zeros(len(alphas))
    alpha_counts = np.zeros(len(alphas))

    for inner_train_idx, inner_val_idx in inner_kf.split(expr_train, y_train['event']):
        X_itr, X_ival = expr_train.values[inner_train_idx], expr_train.values[inner_val_idx]
        y_itr, y_ival = y_train[inner_train_idx], y_train[inner_val_idx]

        try:
            inner_model = CoxnetSurvivalAnalysis(
                l1_ratio=LASSO_L1_RATIO, alphas=alphas, max_iter=100000)
            inner_model.fit(X_itr, y_itr)
            for a_idx in range(len(alphas)):
                try:
                    risk = X_ival @ inner_model.coef_[:, a_idx]
                    ci = concordance_index_censored(
                        y_ival['event'], y_ival['time'], risk)[0]
                    alpha_scores[a_idx] += ci
                    alpha_counts[a_idx] += 1
                except Exception:
                    pass
        except Exception:
            continue

    valid = alpha_counts > 0
    mean_scores = np.where(valid, alpha_scores / np.maximum(alpha_counts, 1), -np.inf)
    best_alpha_idx = int(np.argmax(mean_scores))

    coefs = coxnet.coef_[:, best_alpha_idx]
    selected_genes = [g for g, c in zip(expr_train.columns, coefs) if c != 0]

    return selected_genes, best_alpha_idx, mean_scores[best_alpha_idx]


def select_dysreg_genes_infold(dysreg_train, y_train, n_select=N_DYSREG_SELECT):
    """
    Unchanged from your original correct logic: top-N genes by
    univariate Cox p-value, fit on training fold only.
    """
    cox_pvals = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({
                'T': y_train['time'], 'E': y_train['event'],
                'gene': dysreg_train[gene].values
            })
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals[gene] = cph.summary['p'].values[0]
        except Exception:
            cox_pvals[gene] = 1.0
    top_genes = list(pd.Series(cox_pvals).nsmallest(n_select).index)
    return top_genes


# ------------------------------------------------------------
# SINGLE-SEED SANITY CHECK (seed=42, 5 folds) — confirms the
# in-fold pipeline runs correctly before Block 5's full repeat.
# ------------------------------------------------------------
print("Running single-seed sanity check (seed=42)...")
print("This will be SLOWER than before — Lasso alpha search now")
print("runs fresh per fold instead of loading a pickle.\n")

kf_check = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"{'Fold':<6}{'N_expr_genes':<14}{'N_dysreg':<10}{'Train C-idx':<14}{'Test C-idx':<12}{'Gap':<8}")
print("-" * 66)

check_train_ci, check_test_ci = [], []

for fold, (train_idx, test_idx) in enumerate(kf_check.split(expr, y['event']), 1):
    expr_train, expr_test = expr.iloc[train_idx], expr.iloc[test_idx]
    dysreg_train, dysreg_test = dysreg.iloc[train_idx], dysreg.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # -- Expression: Lasso fit INSIDE fold --
    expr_genes, alpha_idx, inner_ci = select_expression_genes_infold(expr_train, y_train)
    if len(expr_genes) == 0:
        print(f"Fold {fold}: WARNING — 0 genes selected, skipping fold")
        continue

    expr_train_sel = expr_train[expr_genes].copy()
    expr_train_sel.columns = [f"{g}_expr" for g in expr_genes]
    expr_test_sel = expr_test[expr_genes].copy()
    expr_test_sel.columns = [f"{g}_expr" for g in expr_genes]

    # -- Dysreg: top-20 by p-value, INSIDE fold (unchanged) --
    dysreg_genes = select_dysreg_genes_infold(dysreg_train, y_train)
    dysreg_train_sel = dysreg_train[dysreg_genes].copy()
    dysreg_train_sel.columns = [f"{g}_dysreg" for g in dysreg_genes]
    dysreg_test_sel = dysreg_test[dysreg_genes].copy()
    dysreg_test_sel.columns = [f"{g}_dysreg" for g in dysreg_genes]

    # -- Assemble full feature matrix --
    X_train = pd.concat([
        expr_train_sel, dysreg_train_sel,
        immune_features.iloc[train_idx], clinical_features.iloc[train_idx],
        interactions.iloc[train_idx]
    ], axis=1).fillna(0)
    X_test = pd.concat([
        expr_test_sel, dysreg_test_sel,
        immune_features.iloc[test_idx], clinical_features.iloc[test_idx],
        interactions.iloc[test_idx]
    ], axis=1).fillna(0)

    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    model = GradientBoostingSurvivalAnalysis(**GBM_PARAMS, random_state=42)
    model.fit(X_train_s, y_train)

    risk_train = model.predict(X_train_s)
    risk_test = model.predict(X_test_s)

    ci_train = concordance_index_censored(y_train['event'], y_train['time'], risk_train)[0]
    ci_test = concordance_index_censored(y_test['event'], y_test['time'], risk_test)[0]
    check_train_ci.append(ci_train)
    check_test_ci.append(ci_test)

    print(f"{fold:<6}{len(expr_genes):<14}{len(dysreg_genes):<10}{ci_train:.4f}        {ci_test:.4f}       {ci_train-ci_test:+.4f}")

print("-" * 66)
print(f"\nSanity-check mean: train={np.mean(check_train_ci):.4f}  test={np.mean(check_test_ci):.4f}")
print("If this ran without errors, we're ready for Block 5 (full repeated CV).")

Running single-seed sanity check (seed=42)...
This will be SLOWER than before — Lasso alpha search now
runs fresh per fold instead of loading a pickle.

Fold  N_expr_genes  N_dysreg  Train C-idx   Test C-idx  Gap     
------------------------------------------------------------------
1     31            20        0.9095        0.6212       +0.2882
2     45            20        0.9167        0.7503       +0.1663
3     111           20        0.9313        0.6165       +0.3149
4     14            20        0.9027        0.6199       +0.2827
5     39            20        0.8995        0.5873       +0.3122
------------------------------------------------------------------

Sanity-check mean: train=0.9119  test=0.6391
If this ran without errors, we're ready for Block 5 (full repeated CV).


In [7]:
# ============================================================
# Block 4b: Stabilized gene selection + harder GBM regularization
# Replaces the alpha-search and GBM_PARAMS from Block 4/1.
# ============================================================

def select_expression_genes_infold_stable(expr_train, y_train, n_repeats=5, n_inner_folds=5):
    """
    Same idea as before, but averages alpha performance over
    multiple repeated inner CV splits instead of one noisy split.
    This should stop gene counts swinging 14 -> 111 across folds.
    """
    coxnet = CoxnetSurvivalAnalysis(
        l1_ratio=LASSO_L1_RATIO,
        alpha_min_ratio=LASSO_ALPHA_MIN_RATIO,
        n_alphas=LASSO_N_ALPHAS,
        max_iter=100000,
    )
    coxnet.fit(expr_train.values, y_train)
    alphas = coxnet.alphas_

    alpha_scores = np.zeros(len(alphas))
    alpha_counts = np.zeros(len(alphas))

    for repeat in range(n_repeats):
        inner_kf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True,
                                    random_state=1000 + repeat)
        for inner_train_idx, inner_val_idx in inner_kf.split(expr_train, y_train['event']):
            X_itr, X_ival = expr_train.values[inner_train_idx], expr_train.values[inner_val_idx]
            y_itr, y_ival = y_train[inner_train_idx], y_train[inner_val_idx]
            try:
                inner_model = CoxnetSurvivalAnalysis(
                    l1_ratio=LASSO_L1_RATIO, alphas=alphas, max_iter=100000)
                inner_model.fit(X_itr, y_itr)
                for a_idx in range(len(alphas)):
                    try:
                        risk = X_ival @ inner_model.coef_[:, a_idx]
                        ci = concordance_index_censored(
                            y_ival['event'], y_ival['time'], risk)[0]
                        alpha_scores[a_idx] += ci
                        alpha_counts[a_idx] += 1
                    except Exception:
                        pass
            except Exception:
                continue

    valid = alpha_counts > 0
    mean_scores = np.where(valid, alpha_scores / np.maximum(alpha_counts, 1), -np.inf)
    best_alpha_idx = int(np.argmax(mean_scores))

    coefs = coxnet.coef_[:, best_alpha_idx]
    selected_genes = [g for g, c in zip(expr_train.columns, coefs) if c != 0]
    return selected_genes, best_alpha_idx, mean_scores[best_alpha_idx]


# ---- Harder GBM regularization ----
GBM_PARAMS_V2 = dict(
    n_estimators=100,          # was 300 — fewer rounds, less room to memorize
    learning_rate=0.05,
    max_depth=2,
    min_samples_split=30,      # was 20
    min_samples_leaf=15,       # was 10
    subsample=0.7,             # was 0.8 — more row bagging
    max_features=0.6,          # NEW — each tree only sees 60% of columns
)

print("Re-running single-seed sanity check with stabilized selection + harder GBM reg...\n")

kf_check = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
print(f"{'Fold':<6}{'N_expr_genes':<14}{'N_dysreg':<10}{'Train C-idx':<14}{'Test C-idx':<12}{'Gap':<8}")
print("-" * 66)

check_train_ci, check_test_ci = [], []

for fold, (train_idx, test_idx) in enumerate(kf_check.split(expr, y['event']), 1):
    expr_train, expr_test = expr.iloc[train_idx], expr.iloc[test_idx]
    dysreg_train, dysreg_test = dysreg.iloc[train_idx], dysreg.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    expr_genes, alpha_idx, inner_ci = select_expression_genes_infold_stable(expr_train, y_train)
    if len(expr_genes) == 0:
        print(f"Fold {fold}: WARNING — 0 genes selected, skipping fold")
        continue

    expr_train_sel = expr_train[expr_genes].copy()
    expr_train_sel.columns = [f"{g}_expr" for g in expr_genes]
    expr_test_sel = expr_test[expr_genes].copy()
    expr_test_sel.columns = [f"{g}_expr" for g in expr_genes]

    dysreg_genes = select_dysreg_genes_infold(dysreg_train, y_train)
    dysreg_train_sel = dysreg_train[dysreg_genes].copy()
    dysreg_train_sel.columns = [f"{g}_dysreg" for g in dysreg_genes]
    dysreg_test_sel = dysreg_test[dysreg_genes].copy()
    dysreg_test_sel.columns = [f"{g}_dysreg" for g in dysreg_genes]

    X_train = pd.concat([
        expr_train_sel, dysreg_train_sel,
        immune_features.iloc[train_idx], clinical_features.iloc[train_idx],
        interactions.iloc[train_idx]
    ], axis=1).fillna(0)
    X_test = pd.concat([
        expr_test_sel, dysreg_test_sel,
        immune_features.iloc[test_idx], clinical_features.iloc[test_idx],
        interactions.iloc[test_idx]
    ], axis=1).fillna(0)

    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    model = GradientBoostingSurvivalAnalysis(**GBM_PARAMS_V2, random_state=42)
    model.fit(X_train_s, y_train)

    risk_train = model.predict(X_train_s)
    risk_test = model.predict(X_test_s)

    ci_train = concordance_index_censored(y_train['event'], y_train['time'], risk_train)[0]
    ci_test = concordance_index_censored(y_test['event'], y_test['time'], risk_test)[0]
    check_train_ci.append(ci_train)
    check_test_ci.append(ci_test)

    print(f"{fold:<6}{len(expr_genes):<14}{len(dysreg_genes):<10}{ci_train:.4f}        {ci_test:.4f}       {ci_train-ci_test:+.4f}")

print("-" * 66)
print(f"\nStabilized-check mean: train={np.mean(check_train_ci):.4f}  test={np.mean(check_test_ci):.4f}")
print(f"Gap: {np.mean(check_train_ci) - np.mean(check_test_ci):+.4f}")
print("\nCompare to previous unstable run: train=0.9119, test=0.6391, gap=+0.273")

Re-running single-seed sanity check with stabilized selection + harder GBM reg...

Fold  N_expr_genes  N_dysreg  Train C-idx   Test C-idx  Gap     
------------------------------------------------------------------
1     26            20        0.8600        0.5800       +0.2799
2     68            20        0.8755        0.7181       +0.1574
3     60            20        0.8802        0.6271       +0.2532
4     149           20        0.8785        0.6495       +0.2290
5     41            20        0.8615        0.6260       +0.2356
------------------------------------------------------------------

Stabilized-check mean: train=0.8712  test=0.6401
Gap: +0.2310

Compare to previous unstable run: train=0.9119, test=0.6391, gap=+0.273


In [8]:
# ============================================================
# Block 4c: Honest per-stream ablation — same folds, same
# selection logic, same GBM engine. Diagnoses WHERE signal
# is lost rather than tuning blind.
# ============================================================

def run_gbm_cv(X_full_by_fold, y_arr, kf_splits, label):
    """X_full_by_fold: dict fold_idx -> (X_train_df, X_test_df) already scaled"""
    train_cis, test_cis = [], []
    for (train_idx, test_idx), (X_tr, X_te) in zip(kf_splits, X_full_by_fold):
        model = GradientBoostingSurvivalAnalysis(**GBM_PARAMS_V2, random_state=42)
        model.fit(X_tr, y_arr[train_idx])
        risk_tr = model.predict(X_tr)
        risk_te = model.predict(X_te)
        train_cis.append(concordance_index_censored(y_arr[train_idx]['event'], y_arr[train_idx]['time'], risk_tr)[0])
        test_cis.append(concordance_index_censored(y_arr[test_idx]['event'], y_arr[test_idx]['time'], risk_te)[0])
    print(f"{label:<20} train={np.mean(train_cis):.4f}  test={np.mean(test_cis):.4f}  gap={np.mean(train_cis)-np.mean(test_cis):+.4f}")
    return train_cis, test_cis

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(kf.split(expr, y['event']))

print(f"{'Stream':<20}{'':<10}")
print("-" * 60)

# ---- 1. Clinical-only, real Cox baseline ----
cox_test_cis = []
for train_idx, test_idx in splits:
    df_tr = clinical_features.iloc[train_idx].copy()
    df_tr['T'] = y['time'][train_idx]; df_tr['E'] = y['event'][train_idx]
    cph = CoxPHFitter()
    cph.fit(df_tr, duration_col='T', event_col='E', show_progress=False)
    risk_te = cph.predict_partial_hazard(clinical_features.iloc[test_idx])
    ci = concordance_index_censored(y['event'][test_idx], y['time'][test_idx], risk_te)[0]
    cox_test_cis.append(ci)
print(f"{'Clinical-only-Cox':<20}test={np.mean(cox_test_cis):.4f}  (real baseline, not hardcoded 0.700)")

# ---- 2. Clinical-only, GBM (same engine as combined model) ----
splits_data = []
for train_idx, test_idx in splits:
    Xtr, Xte = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns)
    Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns)
    splits_data.append((Xtr_s, Xte_s))
run_gbm_cv(splits_data, y, splits, "Clinical-only-GBM")

# ---- 3. Expression-only ----
splits_data = []
for train_idx, test_idx in splits:
    expr_tr, expr_te = expr.iloc[train_idx], expr.iloc[test_idx]
    genes, _, _ = select_expression_genes_infold_stable(expr_tr, y[train_idx])
    Xtr, Xte = expr_tr[genes], expr_te[genes]
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns)
    Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns)
    splits_data.append((Xtr_s, Xte_s))
run_gbm_cv(splits_data, y, splits, "Expression-only")

# ---- 4. Dysreg-only ----
splits_data = []
for train_idx, test_idx in splits:
    dysreg_tr, dysreg_te = dysreg.iloc[train_idx], dysreg.iloc[test_idx]
    genes = select_dysreg_genes_infold(dysreg_tr, y[train_idx])
    Xtr, Xte = dysreg_tr[genes], dysreg_te[genes]
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns)
    Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns)
    splits_data.append((Xtr_s, Xte_s))
run_gbm_cv(splits_data, y, splits, "Dysreg-only")

# ---- 5. Immune-only ----
splits_data = []
for train_idx, test_idx in splits:
    Xtr, Xte = immune_features.iloc[train_idx], immune_features.iloc[test_idx]
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns)
    Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns)
    splits_data.append((Xtr_s, Xte_s))
run_gbm_cv(splits_data, y, splits, "Immune-only")

print("\nCompare all of the above to Full-combined (already measured): test=0.6401")

Stream                        
------------------------------------------------------------
Clinical-only-Cox   test=0.7036  (real baseline, not hardcoded 0.700)
Clinical-only-GBM    train=0.7233  test=0.6773  gap=+0.0460
Expression-only      train=0.8666  test=0.6222  gap=+0.2444
Dysreg-only          train=0.8113  test=0.6013  gap=+0.2100
Immune-only          train=0.7787  test=0.4745  gap=+0.3043

Compare all of the above to Full-combined (already measured): test=0.6401


In [9]:
# ============================================================
# Block 4d: Single penalized Cox model (elastic net) as
# alternative to GBM — test whether a linear model
# generalizes better at N=470.
# ============================================================

def build_combined_features(train_idx, test_idx, expr_genes, dysreg_genes):
    X_train = pd.concat([
        expr.iloc[train_idx][expr_genes].add_suffix('_expr'),
        dysreg.iloc[train_idx][dysreg_genes].add_suffix('_dysreg'),
        immune_features.iloc[train_idx], clinical_features.iloc[train_idx],
        interactions.iloc[train_idx]
    ], axis=1).fillna(0)
    X_test = pd.concat([
        expr.iloc[test_idx][expr_genes].add_suffix('_expr'),
        dysreg.iloc[test_idx][dysreg_genes].add_suffix('_dysreg'),
        immune_features.iloc[test_idx], clinical_features.iloc[test_idx],
        interactions.iloc[test_idx]
    ], axis=1).fillna(0)
    return X_train, X_test

print(f"{'Coxnet-combined':<20}")
print("-" * 60)

coxnet_train_ci, coxnet_test_ci = [], []

for train_idx, test_idx in splits:
    expr_tr = expr.iloc[train_idx]
    genes, _, _ = select_expression_genes_infold_stable(expr_tr, y[train_idx])
    dysreg_genes = select_dysreg_genes_infold(dysreg.iloc[train_idx], y[train_idx])

    X_train, X_test = build_combined_features(train_idx, test_idx, genes, dysreg_genes)
    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    # l1_ratio < 1 blends L1 (selection) + L2 (stability/shrinkage) —
    # more stable than pure Lasso on correlated genomic features
    coxnet = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.01, n_alphas=50, max_iter=100000)
    coxnet.fit(X_train_s.values, y[train_idx])

    # pick alpha via inner CV on train only (same principle as Block 4)
    best_idx = coxnet.alphas_.argmax() * 0  # placeholder — reuse select_expression_genes_infold_stable's
                                              # inner-CV alpha logic here, or simplify to mid-path alpha
    mid_idx = len(coxnet.alphas_) // 2
    risk_train = X_train_s.values @ coxnet.coef_[:, mid_idx]
    risk_test = X_test_s.values @ coxnet.coef_[:, mid_idx]

    ci_tr = concordance_index_censored(y[train_idx]['event'], y[train_idx]['time'], risk_train)[0]
    ci_te = concordance_index_censored(y[test_idx]['event'], y[test_idx]['time'], risk_test)[0]
    coxnet_train_ci.append(ci_tr); coxnet_test_ci.append(ci_te)

print(f"Coxnet-combined      train={np.mean(coxnet_train_ci):.4f}  test={np.mean(coxnet_test_ci):.4f}  gap={np.mean(coxnet_train_ci)-np.mean(coxnet_test_ci):+.4f}")
print(f"\nCompare: Clinical-only-Cox=0.7036, Full-combined-GBM=0.6401")

Coxnet-combined     
------------------------------------------------------------
Coxnet-combined      train=0.9055  test=0.6624  gap=+0.2431

Compare: Clinical-only-Cox=0.7036, Full-combined-GBM=0.6401


In [10]:
# ============================================================
# Block 4e: Coxnet with PROPER in-fold alpha selection
# (replaces the mid_idx placeholder — that run wasn't a fair test)
# ============================================================

def select_alpha_infold(X_train, y_train, l1_ratio=0.5, n_inner_folds=5, n_repeats=3):
    """
    Fit full alpha path once, then score each alpha via repeated
    inner CV on training data only. Returns best alpha's coef vector.
    """
    coxnet = CoxnetSurvivalAnalysis(
        l1_ratio=l1_ratio, alpha_min_ratio=0.01, n_alphas=50, max_iter=100000)
    coxnet.fit(X_train.values, y_train)
    alphas = coxnet.alphas_

    alpha_scores = np.zeros(len(alphas))
    alpha_counts = np.zeros(len(alphas))

    for repeat in range(n_repeats):
        inner_kf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True,
                                    random_state=2000 + repeat)
        for itr_idx, ival_idx in inner_kf.split(X_train, y_train['event']):
            X_itr, X_ival = X_train.values[itr_idx], X_train.values[ival_idx]
            y_itr, y_ival = y_train[itr_idx], y_train[ival_idx]
            try:
                inner_model = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alphas=alphas, max_iter=100000)
                inner_model.fit(X_itr, y_itr)
                for a_idx in range(len(alphas)):
                    try:
                        risk = X_ival @ inner_model.coef_[:, a_idx]
                        ci = concordance_index_censored(y_ival['event'], y_ival['time'], risk)[0]
                        alpha_scores[a_idx] += ci
                        alpha_counts[a_idx] += 1
                    except Exception:
                        pass
            except Exception:
                continue

    valid = alpha_counts > 0
    mean_scores = np.where(valid, alpha_scores / np.maximum(alpha_counts, 1), -np.inf)
    best_idx = int(np.argmax(mean_scores))
    return coxnet.coef_[:, best_idx], mean_scores[best_idx], best_idx, len(alphas)


print(f"{'Coxnet-combined-tuned':<24}")
print("-" * 66)
print(f"{'Fold':<6}{'N_nonzero':<12}{'AlphaIdx/Total':<16}{'Train C-idx':<14}{'Test C-idx':<12}{'Gap':<8}")
print("-" * 66)

coxnet_train_ci, coxnet_test_ci = [], []

for fold, (train_idx, test_idx) in enumerate(splits, 1):
    expr_tr = expr.iloc[train_idx]
    genes, _, _ = select_expression_genes_infold_stable(expr_tr, y[train_idx])
    dysreg_genes = select_dysreg_genes_infold(dysreg.iloc[train_idx], y[train_idx])

    X_train, X_test = build_combined_features(train_idx, test_idx, genes, dysreg_genes)
    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    coefs, inner_score, best_idx, n_alphas = select_alpha_infold(X_train_s, y[train_idx])
    n_nonzero = int((coefs != 0).sum())

    risk_train = X_train_s.values @ coefs
    risk_test = X_test_s.values @ coefs

    ci_tr = concordance_index_censored(y[train_idx]['event'], y[train_idx]['time'], risk_train)[0]
    ci_te = concordance_index_censored(y[test_idx]['event'], y[test_idx]['time'], risk_test)[0]
    coxnet_train_ci.append(ci_tr); coxnet_test_ci.append(ci_te)

    print(f"{fold:<6}{n_nonzero:<12}{best_idx}/{n_alphas:<12}{ci_tr:.4f}        {ci_te:.4f}       {ci_tr-ci_te:+.4f}")

print("-" * 66)
print(f"\nCoxnet-combined-tuned mean: train={np.mean(coxnet_train_ci):.4f}  test={np.mean(coxnet_test_ci):.4f}  gap={np.mean(coxnet_train_ci)-np.mean(coxnet_test_ci):+.4f}")
print(f"\nCompare:")
print(f"  Clinical-only-Cox:        0.7036")
print(f"  Full-combined-GBM:        0.6401")
print(f"  Coxnet-combined (crude):  0.6624")
print(f"  Coxnet-combined (tuned):  {np.mean(coxnet_test_ci):.4f}")

Coxnet-combined-tuned   
------------------------------------------------------------------
Fold  N_nonzero   AlphaIdx/Total  Train C-idx   Test C-idx  Gap     
------------------------------------------------------------------
1     49          24/50          0.8446        0.6783       +0.1663
2     95          32/50          0.9340        0.7450       +0.1891
3     85          30/50          0.9355        0.6047       +0.3308
4     162         49/50          0.9999        0.6838       +0.3161
5     62          26/50          0.8786        0.5317       +0.3469
------------------------------------------------------------------

Coxnet-combined-tuned mean: train=0.9185  test=0.6487  gap=+0.2698

Compare:
  Clinical-only-Cox:        0.7036
  Full-combined-GBM:        0.6401
  Coxnet-combined (crude):  0.6624
  Coxnet-combined (tuned):  0.6487


In [11]:
# ============================================================
# Block 5: Repeated CV — 5 folds x 10 seeds = 50 fits.
# FROZEN config: single GBM, stabilized in-fold selection.
# No further tuning after this — this is the number we report.
# ============================================================

all_results = []   # one row per (seed, fold)

print(f"Running {N_FOLDS} folds x {N_SEEDS} seeds = {N_FOLDS * N_SEEDS} total fits...")
print("This will take a while — in-fold Lasso alpha search runs fresh every time.\n")

oof_risk_by_seed = {}   # seed -> array of OOF risk scores, aligned to `common`

for seed in SEEDS:
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_risk = np.full(len(common), np.nan)

    for fold, (train_idx, test_idx) in enumerate(kf.split(expr, y['event']), 1):
        expr_tr = expr.iloc[train_idx]
        genes, _, _ = select_expression_genes_infold_stable(expr_tr, y[train_idx])
        dysreg_genes = select_dysreg_genes_infold(dysreg.iloc[train_idx], y[train_idx])

        if len(genes) == 0:
            print(f"  seed={seed} fold={fold}: WARNING 0 expr genes selected, skipping")
            continue

        X_train, X_test = build_combined_features(train_idx, test_idx, genes, dysreg_genes)
        scaler = StandardScaler()
        X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
        X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

        model = GradientBoostingSurvivalAnalysis(**GBM_PARAMS_V2, random_state=seed)
        model.fit(X_train_s, y[train_idx])

        risk_train = model.predict(X_train_s)
        risk_test = model.predict(X_test_s)
        oof_risk[test_idx] = risk_test

        ci_train = concordance_index_censored(y[train_idx]['event'], y[train_idx]['time'], risk_train)[0]
        ci_test = concordance_index_censored(y[test_idx]['event'], y[test_idx]['time'], risk_test)[0]

        all_results.append({
            'seed': seed, 'fold': fold,
            'n_expr_genes': len(genes), 'n_dysreg_genes': len(dysreg_genes),
            'train_ci': ci_train, 'test_ci': ci_test, 'gap': ci_train - ci_test
        })

    oof_risk_by_seed[seed] = oof_risk.copy()
    seed_test_cis = [r['test_ci'] for r in all_results if r['seed'] == seed]
    print(f"seed={seed}  mean test C-idx this seed: {np.mean(seed_test_cis):.4f}")

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 60)
print("REPEATED CV SUMMARY (50 fits)")
print("=" * 60)
print(f"Test C-index:  mean={results_df['test_ci'].mean():.4f}  std={results_df['test_ci'].std():.4f}")
print(f"Train C-index: mean={results_df['train_ci'].mean():.4f}  std={results_df['train_ci'].std():.4f}")
print(f"Mean gap:      {results_df['gap'].mean():+.4f}")
print(f"\nGene selection stability across all 50 fits:")
print(f"  N expression genes: min={results_df['n_expr_genes'].min()}, max={results_df['n_expr_genes'].max()}, mean={results_df['n_expr_genes'].mean():.1f}")
print(f"  N dysreg genes: {results_df['n_dysreg_genes'].unique()} (should always be {N_DYSREG_SELECT})")

print(f"\nPer-seed test C-index (checking seed-to-seed stability):")
print(results_df.groupby('seed')['test_ci'].mean())

print(f"\nCompare to real clinical baseline: 0.7036")

Running 5 folds x 10 seeds = 50 total fits...
This will take a while — in-fold Lasso alpha search runs fresh every time.

seed=42  mean test C-idx this seed: 0.6401
seed=43  mean test C-idx this seed: 0.6453
seed=44  mean test C-idx this seed: 0.6234
seed=45  mean test C-idx this seed: 0.6448
seed=46  mean test C-idx this seed: 0.6285
seed=47  mean test C-idx this seed: 0.6311
seed=48  mean test C-idx this seed: 0.6633
seed=49  mean test C-idx this seed: 0.6424
seed=50  mean test C-idx this seed: 0.7000
seed=51  mean test C-idx this seed: 0.6429

REPEATED CV SUMMARY (50 fits)
Test C-index:  mean=0.6462  std=0.0585
Train C-index: mean=0.8665  std=0.0148
Mean gap:      +0.2203

Gene selection stability across all 50 fits:
  N expression genes: min=5, max=149, mean=51.0
  N dysreg genes: [20] (should always be 20)

Per-seed test C-index (checking seed-to-seed stability):
seed
42    0.640142
43    0.645342
44    0.623411
45    0.644774
46    0.628520
47    0.631093
48    0.663291
49    0.6

In [12]:
# ============================================================
# Block 6: Pooled OOF C-index + bootstrap CI.
# This is the number that goes in the paper — not the
# fold-average from Block 5.
# ============================================================

def bootstrap_ci(risk, times, events, n_boot=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    n = len(risk)
    boot_scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        try:
            ci = concordance_index_censored(events[idx], times[idx], risk[idx])[0]
            boot_scores.append(ci)
        except Exception:
            continue
    boot_scores = np.array(boot_scores)
    return np.mean(boot_scores), np.percentile(boot_scores, 2.5), np.percentile(boot_scores, 97.5)

# ---- Pooled OOF C-index per seed, then averaged ----
pooled_cis = []
for seed in SEEDS:
    oof = oof_risk_by_seed[seed]
    valid = ~np.isnan(oof)
    ci = concordance_index_censored(y['event'][valid], y['time'][valid], oof[valid])[0]
    pooled_cis.append(ci)

print("Pooled OOF C-index per seed:")
for seed, ci in zip(SEEDS, pooled_cis):
    print(f"  seed={seed}: {ci:.4f}")

pooled_mean = np.mean(pooled_cis)
pooled_std = np.std(pooled_cis)
print(f"\nPooled OOF C-index across seeds: {pooled_mean:.4f} +/- {pooled_std:.4f}")

# ---- Bootstrap CI using the single best-representative seed's OOF risk ----
# (using seed=42 as the "primary" reported OOF vector, consistent with your
#  original convention; bootstrap over patients quantifies sampling
#  uncertainty, not seed variability, so the two are reported separately)
primary_oof = oof_risk_by_seed[42]
valid = ~np.isnan(primary_oof)
boot_mean, boot_lo, boot_hi = bootstrap_ci(
    primary_oof[valid], y['time'][valid], y['event'][valid])

print(f"\nBootstrap 95% CI (seed=42 OOF, {N_BOOTSTRAP} resamples):")
print(f"  C-index: {boot_mean:.4f}  [{boot_lo:.4f}, {boot_hi:.4f}]")

# ---- Clinical baseline: same treatment for fair comparison ----
clinical_oof = np.full(len(common), np.nan)
kf42 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for train_idx, test_idx in kf42.split(clinical_features, y['event']):
    df_tr = clinical_features.iloc[train_idx].copy()
    df_tr['T'] = y['time'][train_idx]; df_tr['E'] = y['event'][train_idx]
    cph = CoxPHFitter()
    cph.fit(df_tr, duration_col='T', event_col='E', show_progress=False)
    clinical_oof[test_idx] = cph.predict_partial_hazard(clinical_features.iloc[test_idx]).values

clin_boot_mean, clin_boot_lo, clin_boot_hi = bootstrap_ci(
    clinical_oof, y['time'], y['event'])

print(f"\nClinical baseline bootstrap 95% CI (same seed=42 splits):")
print(f"  C-index: {clin_boot_mean:.4f}  [{clin_boot_lo:.4f}, {clin_boot_hi:.4f}]")

print("\n" + "=" * 60)
print("FINAL HEADLINE COMPARISON")
print("=" * 60)
print(f"Molecular combined model: {boot_mean:.4f}  [{boot_lo:.4f}, {boot_hi:.4f}]")
print(f"Clinical baseline:        {clin_boot_mean:.4f}  [{clin_boot_lo:.4f}, {clin_boot_hi:.4f}]")
overlap = boot_hi >= clin_boot_lo
print(f"\nCIs overlap: {overlap}  (if True, difference may not be statistically meaningful)")

Pooled OOF C-index per seed:
  seed=42: 0.6417
  seed=43: 0.6477
  seed=44: 0.6291
  seed=45: 0.6344
  seed=46: 0.6300
  seed=47: 0.6285
  seed=48: 0.6657
  seed=49: 0.6364
  seed=50: 0.6939
  seed=51: 0.6401

Pooled OOF C-index across seeds: 0.6447 +/- 0.0195

Bootstrap 95% CI (seed=42 OOF, 1000 resamples):
  C-index: 0.6425  [0.5816, 0.7051]

Clinical baseline bootstrap 95% CI (same seed=42 splits):
  C-index: 0.6973  [0.6392, 0.7590]

FINAL HEADLINE COMPARISON
Molecular combined model: 0.6425  [0.5816, 0.7051]
Clinical baseline:        0.6973  [0.6392, 0.7590]

CIs overlap: True  (if True, difference may not be statistically meaningful)


In [13]:
# ============================================================
# Block 6b: Paired bootstrap on (clinical - molecular) difference.
# More rigorous than comparing independent CIs — resamples the
# SAME patients for both models each iteration, so it isolates
# whether clinical genuinely outperforms molecular, not just
# whether their CIs happen to overlap.
# ============================================================

def paired_bootstrap_diff(risk_a, risk_b, times, events, n_boot=N_BOOTSTRAP, seed=0):
    """risk_a - risk_b difference in C-index, per resample."""
    rng = np.random.default_rng(seed)
    n = len(risk_a)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        try:
            ci_a = concordance_index_censored(events[idx], times[idx], risk_a[idx])[0]
            ci_b = concordance_index_censored(events[idx], times[idx], risk_b[idx])[0]
            diffs.append(ci_a - ci_b)
        except Exception:
            continue
    diffs = np.array(diffs)
    return diffs

# Use identical seed=42 splits already computed for both models
diffs = paired_bootstrap_diff(clinical_oof, primary_oof, y['time'], y['event'])

diff_mean = diffs.mean()
diff_lo, diff_hi = np.percentile(diffs, [2.5, 97.5])
pct_favoring_clinical = (diffs > 0).mean()

print("Paired bootstrap: Clinical C-index minus Molecular C-index")
print(f"  Mean difference: {diff_mean:+.4f}")
print(f"  95% CI on difference: [{diff_lo:+.4f}, {diff_hi:+.4f}]")
print(f"  Proportion of resamples favoring clinical: {pct_favoring_clinical*100:.1f}%")
print(f"\n  Does the difference CI exclude 0? {'YES — clinical significantly better' if diff_lo > 0 else 'NO — cannot claim significant difference'}")

Paired bootstrap: Clinical C-index minus Molecular C-index
  Mean difference: +0.0548
  95% CI on difference: [-0.0074, +0.1210]
  Proportion of resamples favoring clinical: 95.3%

  Does the difference CI exclude 0? NO — cannot claim significant difference


In [14]:
# ============================================================
# Block 6c: Confirm bootstrap stability at higher resolution
# ============================================================

diffs_5k = paired_bootstrap_diff(clinical_oof, primary_oof, y['time'], y['event'],
                                   n_boot=5000, seed=1)

diff_mean_5k = diffs_5k.mean()
diff_lo_5k, diff_hi_5k = np.percentile(diffs_5k, [2.5, 97.5])
pct_favoring_clinical_5k = (diffs_5k > 0).mean()

print("Paired bootstrap (n=5000): Clinical minus Molecular C-index")
print(f"  Mean difference: {diff_mean_5k:+.4f}")
print(f"  95% CI on difference: [{diff_lo_5k:+.4f}, {diff_hi_5k:+.4f}]")
print(f"  Proportion favoring clinical: {pct_favoring_clinical_5k*100:.1f}%")

print(f"\nCompare to n=1000 run:")
print(f"  Mean diff: +0.0548 -> {diff_mean_5k:+.4f}")
print(f"  CI: [-0.0074, +0.1210] -> [{diff_lo_5k:+.4f}, {diff_hi_5k:+.4f}]")
print(f"  Pct favoring clinical: 95.3% -> {pct_favoring_clinical_5k*100:.1f}%")

stable = abs(diff_mean_5k - 0.0548) < 0.01 and abs(diff_lo_5k - (-0.0074)) < 0.015
print(f"\nEstimate stable across resample counts: {stable}")

Paired bootstrap (n=5000): Clinical minus Molecular C-index
  Mean difference: +0.0572
  95% CI on difference: [-0.0056, +0.1173]
  Proportion favoring clinical: 96.4%

Compare to n=1000 run:
  Mean diff: +0.0548 -> +0.0572
  CI: [-0.0074, +0.1210] -> [-0.0056, +0.1173]
  Pct favoring clinical: 95.3% -> 96.4%

Estimate stable across resample counts: True


In [15]:
# ============================================================
# Block 8: Within-stage analysis — honest, using OOF risk
# scores from the frozen, leakage-free pipeline (seed=42,
# consistent with primary_oof used in Block 6).
#
# Compares: does the molecular model discriminate survival
# WITHIN a stage, better than clinical alone can (which by
# definition can't discriminate within a single stage group
# using stage itself)?
# ============================================================

MIN_PATIENTS_PER_STAGE = 15   # up from the original >5 guard —
                                # too small a subgroup gives an
                                # uninterpretable C-index regardless
                                # of the point estimate
MIN_EVENTS_PER_STAGE = 5

stage_groups = clinical['stage_group'].values

print(f"{'Stage':<12}{'N':<6}{'Events':<8}{'Molecular C-idx':<20}{'95% CI':<20}{'Clinical C-idx (age/gender only)':<20}")
print("-" * 90)

within_stage_results = {}

for stage in ['Stage I', 'Stage II', 'Stage III', 'Stage IV']:
    mask = stage_groups == stage
    n_stage = mask.sum()
    n_events = y['event'][mask].sum()

    if n_stage < MIN_PATIENTS_PER_STAGE or n_events < MIN_EVENTS_PER_STAGE:
        print(f"{stage:<12}{n_stage:<6}{n_events:<8}SKIPPED (n<{MIN_PATIENTS_PER_STAGE} or events<{MIN_EVENTS_PER_STAGE})")
        continue

    # Molecular model's OOF risk, within this stage subgroup
    risk_mol = primary_oof[mask]
    times_s = y['time'][mask]
    events_s = y['event'][mask]

    ci_mol = concordance_index_censored(events_s, times_s, risk_mol)[0]
    boot_mean, boot_lo, boot_hi = bootstrap_ci(risk_mol, times_s, events_s, n_boot=2000, seed=42)

    # "Clinical, within-stage" baseline: age + gender only (stage itself
    # is constant within the subgroup, so it can't be used here — this
    # is the correct within-stage clinical comparator, not the full
    # clinical_oof which includes stage dummies)
    age_gender = clinical_features[['age', 'gender']].values[mask]
    df_tmp = pd.DataFrame({'age': age_gender[:, 0], 'gender': age_gender[:, 1],
                            'T': times_s, 'E': events_s})
    try:
        cph_stage = CoxPHFitter()
        cph_stage.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        risk_clin_stage = cph_stage.predict_partial_hazard(df_tmp[['age', 'gender']]).values
        ci_clin_stage = concordance_index_censored(events_s, times_s, risk_clin_stage)[0]
    except Exception:
        ci_clin_stage = np.nan

    within_stage_results[stage] = {
        'n': int(n_stage), 'events': int(n_events),
        'molecular_ci': round(float(ci_mol), 4),
        'molecular_ci_95CI': [round(float(boot_lo), 4), round(float(boot_hi), 4)],
        'clinical_age_gender_ci': round(float(ci_clin_stage), 4) if not np.isnan(ci_clin_stage) else None
    }

    print(f"{stage:<12}{n_stage:<6}{n_events:<8}{ci_mol:<20.4f}[{boot_lo:.3f}, {boot_hi:.3f}]{'':<6}{ci_clin_stage:.4f}")

print("\n" + "=" * 60)
print("COMPARISON TO ORIGINAL (INFLATED) DOCX CLAIMS")
print("=" * 60)
print("Docx claimed: Stage I molecular=0.888 vs clinical=0.619")
print("Docx claimed: Stage III molecular=0.890 vs clinical=0.567")
print("(those used in-sample / pre-leak-fix scoring — compare honestly above)")

print(f"\nWithin-stage results dict (for JSON export in Block 10):")
print(json.dumps(within_stage_results, indent=2))

Stage       N     Events  Molecular C-idx     95% CI              Clinical C-idx (age/gender only)
------------------------------------------------------------------------------------------
Stage I     256   39      0.5207              [0.400, 0.645]      0.6266
Stage II    112   34      0.5680              [0.468, 0.680]      0.4767
Stage III   77    35      0.6035              [0.463, 0.733]      0.5925
Stage IV    25    11      0.7033              [0.457, 0.913]      0.7747

COMPARISON TO ORIGINAL (INFLATED) DOCX CLAIMS
Docx claimed: Stage I molecular=0.888 vs clinical=0.619
Docx claimed: Stage III molecular=0.890 vs clinical=0.567
(those used in-sample / pre-leak-fix scoring — compare honestly above)

Within-stage results dict (for JSON export in Block 10):
{
  "Stage I": {
    "n": 256,
    "events": 39,
    "molecular_ci": 0.5207,
    "molecular_ci_95CI": [
      0.4004,
      0.6446
    ],
    "clinical_age_gender_ci": 0.6266
  },
  "Stage II": {
    "n": 112,
    "events": 34,


In [16]:
# ============================================================
# Block 9: Gene selection stability diagnostic.
# Quantifies how consistent the in-fold Lasso selection is
# across the 50 fits from Block 5 — turns "genes ranged 5-149"
# into a real, citable stability metric.
# ============================================================

# We need to re-run selection while STORING the actual gene sets
# (Block 5 only stored counts, not the gene lists themselves)

gene_sets_by_fit = []   # list of sets, one per (seed, fold)

print("Re-collecting gene sets across all 50 fits for stability analysis...")
print("(reusing the same seeds/folds as Block 5 — no new modeling, just tracking selections)\n")

for seed in SEEDS:
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (train_idx, test_idx) in enumerate(kf.split(expr, y['event']), 1):
        expr_tr = expr.iloc[train_idx]
        genes, _, _ = select_expression_genes_infold_stable(expr_tr, y[train_idx])
        gene_sets_by_fit.append(set(genes))

n_fits = len(gene_sets_by_fit)
print(f"Collected {n_fits} gene sets.\n")

# ---- Pairwise Jaccard similarity across all fits ----
jaccard_scores = []
for i in range(n_fits):
    for j in range(i + 1, n_fits):
        a, b = gene_sets_by_fit[i], gene_sets_by_fit[j]
        union = len(a | b)
        inter = len(a & b)
        jaccard = inter / union if union > 0 else 0.0
        jaccard_scores.append(jaccard)

jaccard_scores = np.array(jaccard_scores)
print(f"Pairwise Jaccard similarity across all {n_fits} gene sets ({len(jaccard_scores)} pairs):")
print(f"  Mean: {jaccard_scores.mean():.4f}")
print(f"  Median: {np.median(jaccard_scores):.4f}")
print(f"  Std: {jaccard_scores.std():.4f}")
print(f"  Min: {jaccard_scores.min():.4f}  Max: {jaccard_scores.max():.4f}")

# ---- Which genes appear most consistently? ----
from collections import Counter
gene_appearance_counts = Counter()
for gene_set in gene_sets_by_fit:
    for gene in gene_set:
        gene_appearance_counts[gene] += 1

top_stable_genes = gene_appearance_counts.most_common(20)
print(f"\nTop 20 most consistently selected genes (out of {n_fits} fits):")
for gene, count in top_stable_genes:
    pct = count / n_fits * 100
    print(f"  {gene}: {count}/{n_fits} fits ({pct:.1f}%)")

n_genes_never_below_50pct = sum(1 for g, c in gene_appearance_counts.items() if c / n_fits >= 0.5)
print(f"\nGenes selected in >=50% of fits: {n_genes_never_below_50pct}")
print(f"Total unique genes ever selected across all fits: {len(gene_appearance_counts)}")

stability_summary = {
    "n_fits": n_fits,
    "gene_count_range": [int(results_df['n_expr_genes'].min()), int(results_df['n_expr_genes'].max())],
    "gene_count_mean": round(float(results_df['n_expr_genes'].mean()), 1),
    "pairwise_jaccard_mean": round(float(jaccard_scores.mean()), 4),
    "pairwise_jaccard_median": round(float(np.median(jaccard_scores)), 4),
    "genes_selected_in_50pct_or_more_fits": n_genes_never_below_50pct,
    "total_unique_genes_ever_selected": len(gene_appearance_counts),
    "top_10_stable_genes": [{"gene": g, "fits_present": c, "pct": round(c/n_fits*100, 1)} for g, c in top_stable_genes[:10]]
}

print(f"\nStability summary (for JSON export):")
print(json.dumps(stability_summary, indent=2))

Re-collecting gene sets across all 50 fits for stability analysis...
(reusing the same seeds/folds as Block 5 — no new modeling, just tracking selections)

Collected 50 gene sets.

Pairwise Jaccard similarity across all 50 gene sets (1225 pairs):
  Mean: 0.2152
  Median: 0.2254
  Std: 0.0954
  Min: 0.0125  Max: 0.4677

Top 20 most consistently selected genes (out of 50 fits):
  HOXB9: 46/50 fits (92.0%)
  DKK1: 45/50 fits (90.0%)
  NLRP2: 44/50 fits (88.0%)
  NTS: 42/50 fits (84.0%)
  CLEC18A: 39/50 fits (78.0%)
  LTK: 38/50 fits (76.0%)
  C1QL2: 37/50 fits (74.0%)
  EREG: 35/50 fits (70.0%)
  IGFBP1: 34/50 fits (68.0%)
  C20orf114: 34/50 fits (68.0%)
  EPGN: 33/50 fits (66.0%)
  TMEM139: 31/50 fits (62.0%)
  PKHD1L1: 31/50 fits (62.0%)
  TMEM215: 30/50 fits (60.0%)
  NCAM2: 29/50 fits (58.0%)
  BPIL1: 29/50 fits (58.0%)
  MS4A1: 28/50 fits (56.0%)
  SPRR1B: 28/50 fits (56.0%)
  CRHR2: 28/50 fits (56.0%)
  COMP: 27/50 fits (54.0%)

Genes selected in >=50% of fits: 25
Total unique genes

In [17]:
# ============================================================
# Block 10: Final save. All numbers pulled from actual computed
# results — nothing hardcoded, unlike prior versions.
# ============================================================

final_results = {
    "model": "Single regularised Gradient Boosting Survival Model (GBM), leakage-free",
    "notebook": "12_primary_model_leakagefree_FINAL.ipynb",
    "cohort": {
        "n_patients": int(len(common)),
        "n_events": int(y['event'].sum()),
        "event_rate": round(float(y['event'].mean()), 4),
        "excluded_missing_stage": 8,
        "note": "8 patients excluded for missing AJCC stage (raw 'stage' field also NaN — confirmed unrecoverable)"
    },
    "methodology": {
        "cv_strategy": f"StratifiedKFold_{N_FOLDS}fold_x_{N_SEEDS}seeds_repeated",
        "expression_gene_selection": "Cox-Lasso (CoxnetSurvivalAnalysis), alpha selected via nested inner CV (5 folds x 5 repeats), fit fresh INSIDE each outer fold — fixes prior global-selection leak",
        "dysreg_gene_selection": f"Top-{N_DYSREG_SELECT} by univariate Cox p-value, selected fresh inside each outer fold",
        "gbm_params": GBM_PARAMS_V2,
        "leakage_free": True,
        "no_neural_component": True,
        "no_ensemble": True
    },
    "primary_result": {
        "pooled_oof_cindex_seed42": round(float(concordance_index_censored(y['event'][~np.isnan(primary_oof)], y['time'][~np.isnan(primary_oof)], primary_oof[~np.isnan(primary_oof)])[0]), 4),
        "bootstrap_95CI": [round(float(boot_lo), 4), round(float(boot_hi), 4)],
        "repeated_cv_pooled_mean_across_10_seeds": round(float(pooled_mean), 4),
        "repeated_cv_pooled_std_across_10_seeds": round(float(pooled_std), 4),
        "fold_level_mean_test_cindex": round(float(results_df['test_ci'].mean()), 4),
        "fold_level_std_test_cindex": round(float(results_df['test_ci'].std()), 4),
        "mean_train_test_gap": round(float(results_df['gap'].mean()), 4)
    },
    "clinical_baseline": {
        "cindex": round(float(clin_boot_mean), 4),
        "bootstrap_95CI": [round(float(clin_boot_lo), 4), round(float(clin_boot_hi), 4)],
        "features": "age, gender, stage (dummy-coded)"
    },
    "paired_comparison": {
        "clinical_minus_molecular_mean_diff": round(float(diff_mean_5k), 4),
        "diff_95CI": [round(float(diff_lo_5k), 4), round(float(diff_hi_5k), 4)],
        "pct_resamples_favoring_clinical": round(float(pct_favoring_clinical_5k) * 100, 1),
        "significant_at_95pct": bool(diff_lo_5k > 0),
        "conclusion": "Molecular combined model does NOT significantly exceed clinical staging alone. Difference CI includes zero, though point estimate and resample proportion (96.4%) lean toward clinical outperforming."
    },
    "within_stage_analysis": within_stage_results,
    "within_stage_conclusion": "All within-stage molecular C-index bootstrap CIs include 0.5 — no stage group shows statistically defensible within-stage discrimination. This contradicts the prior in-sample estimate (0.888-0.890) reported before leakage correction.",
    "gene_selection_stability": stability_summary,
    "ablation_scoreboard": {
        "clinical_only_cox": 0.7036,
        "clinical_only_gbm": 0.6773,
        "expression_only_gbm": 0.6222,
        "dysreg_only_gbm": 0.6013,
        "immune_only_gbm": 0.4745,
        "full_combined_gbm": 0.6462,
        "coxnet_combined_tuned": 0.6487
    },
    "comparison_to_prior_leaky_results": {
        "FINAL_primary_model_run_py_leaky": 0.693,
        "NB11c_ensemble_leaky": 0.702,
        "this_corrected_result": round(float(pooled_mean), 4),
        "note": "Prior results used a globally-fixed 72-gene set (NB03 pickle) applied identically across all CV folds — genuine selection leakage. This notebook fixes that by re-selecting genes inside every fold."
    },
    "consensus_gene_signature": {
        "description": "Genes selected in >=50% of the 50 repeated-CV fits — a stable candidate set, distinct from the full unstable per-fit selection",
        "genes": [g for g, c in gene_appearance_counts.items() if c / stability_summary['n_fits'] >= 0.5]
    }
}

with open(f'{OUT_MODEL_DIR}/results_PRIMARY_corrected.json', 'w') as f:
    json.dump(final_results, f, indent=2)

# Save OOF predictions CSV (seed=42, the primary reported seed)
oof_df_final = pd.DataFrame({
    'patient_id': common,
    'risk_score_molecular': primary_oof,
    'risk_score_clinical': clinical_oof,
    'time': y['time'],
    'event': y['event'].astype(int),
    'stage_group': clinical['stage_group'].values,
    'age': clinical['age'].values,
    'gender': clinical['gender'].values,
})
oof_df_final.to_csv(f'{OUT_RESULTS_DIR}/oof_predictions_corrected.csv', index=False)

print(f"Saved: {OUT_MODEL_DIR}/results_PRIMARY_corrected.json")
print(f"Saved: {OUT_RESULTS_DIR}/oof_predictions_corrected.csv")
print("\n" + "=" * 65)
print("CORRECTED PRIMARY RESULT")
print("=" * 65)
print(f"Molecular combined:  {pooled_mean:.4f} [{boot_lo:.4f}, {boot_hi:.4f}]")
print(f"Clinical baseline:   {clin_boot_mean:.4f} [{clin_boot_lo:.4f}, {clin_boot_hi:.4f}]")
print(f"Difference (clinical - molecular): {diff_mean_5k:+.4f} [{diff_lo_5k:+.4f}, {diff_hi_5k:+.4f}]")
print(f"Consensus stable gene signature: {len(final_results['consensus_gene_signature']['genes'])} genes")
print("=" * 65)

Saved: /Users/parthshringarpure/Desktop/AI/Projects/luad_survival/models/final_leakagefree_v2/results_PRIMARY_corrected.json
Saved: /Users/parthshringarpure/Desktop/AI/Projects/luad_survival/outputs/results/oof_predictions_corrected.csv

CORRECTED PRIMARY RESULT
Molecular combined:  0.6447 [0.4571, 0.9126]
Clinical baseline:   0.6973 [0.6392, 0.7590]
Difference (clinical - molecular): +0.0572 [-0.0056, +0.1173]
Consensus stable gene signature: 25 genes


In [18]:
# ============================================================
# Block 10 fix: rename within-stage bootstrap variables so they
# never collide with Block 6's pooled boot_lo/boot_hi again,
# then re-save the JSON with correct values.
# ============================================================

# Recompute the CORRECT pooled bootstrap (Block 6's real numbers,
# unaffected by the Block 8 variable collision — Block 6 itself
# printed the right numbers at the time, only later code reused
# the variable names)
correct_boot_mean, correct_boot_lo, correct_boot_hi = bootstrap_ci(
    primary_oof[~np.isnan(primary_oof)],
    y['time'][~np.isnan(primary_oof)],
    y['event'][~np.isnan(primary_oof)],
    n_boot=N_BOOTSTRAP, seed=0)

print("Recomputed pooled molecular bootstrap CI (should match Block 6 output):")
print(f"  {correct_boot_mean:.4f} [{correct_boot_lo:.4f}, {correct_boot_hi:.4f}]")
print("  (Block 6 originally reported: 0.6425 [0.5816, 0.7051])")

# Patch the saved JSON's primary_result with the correct CI
final_results['primary_result']['bootstrap_95CI'] = [
    round(float(correct_boot_lo), 4), round(float(correct_boot_hi), 4)]

with open(f'{OUT_MODEL_DIR}/results_PRIMARY_corrected.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\nJSON re-saved with corrected pooled CI.")
print(f"Molecular combined:  {pooled_mean:.4f} [{correct_boot_lo:.4f}, {correct_boot_hi:.4f}]")
print(f"Clinical baseline:   {clin_boot_mean:.4f} [{clin_boot_lo:.4f}, {clin_boot_hi:.4f}]")

Recomputed pooled molecular bootstrap CI (should match Block 6 output):
  0.6425 [0.5816, 0.7051]
  (Block 6 originally reported: 0.6425 [0.5816, 0.7051])

JSON re-saved with corrected pooled CI.
Molecular combined:  0.6447 [0.5816, 0.7051]
Clinical baseline:   0.6973 [0.6392, 0.7590]
